# N171 · 测试、调试与性能回归

**目标：** 用边界、状态性质和复杂度诊断验证实现。

**先修：** N010, N015, N042, N049, N147。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 最小复现；多解比较；别名；整数/浮点；缓存污染；操作计数；环境隔离。

## 从问题到算法

测试规范化只能消除题意允许的无关差异，不能抹掉错误。组合题允许答案顺序不同，但重复答案通常是错的，所以规范化保留重数；路径题的节点顺序有意义，不能拿同一个排序函数处理。

链表测试要保留节点引用，检查无环、身份和 next 是否恢复；缓存测试除了 get 返回值，还核对每次操作后的完整 LRU 顺序与双向连接。这样才能抓到“暂时输出相同，但内部状态已错”的实现。

性能回归优先统计真正执行的比较或状态数。二分区间每轮至少近似减半，可以导出对数上界；固定硬件上的毫秒门槛既不能证明复杂度，也容易产生偶然失败。

## 最小实现

**本章接口：** `normalize_solution_output(x)`、`assert_linked_structure(head)`、`differential_test_lru(operations)`、`count_binary_search_iterations()`

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def nodes_to_list(head):
    values, seen = [], set()
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values

class _CacheNode:
    def __init__(self,key=None,value=None): self.key=key; self.value=value; self.frequency=1; self.prev=self.next=None

class _NodeList:
    def __init__(self):
        self.head=_CacheNode(); self.tail=_CacheNode(); self.head.next=self.tail; self.tail.prev=self.head; self.size=0
    def append(self,node):
        previous=self.tail.prev; previous.next=node; node.prev=previous; node.next=self.tail; self.tail.prev=node; self.size+=1
    def remove(self,node):
        node.prev.next=node.next; node.next.prev=node.prev; node.prev=node.next=None; self.size-=1
    def pop_left(self):
        if not self.size: raise IndexError('empty node list')
        node=self.head.next; self.remove(node); return node
class LRUCache:
    def __init__(self,capacity):
        if capacity<0: raise ValueError('nonnegative capacity required')
        self.capacity=capacity; self.nodes={}; self.order=_NodeList()
    def _touch(self,node): self.order.remove(node); self.order.append(node)
    def get(self,key):
        if key not in self.nodes: return -1
        node=self.nodes[key]; self._touch(node); return node.value
    def put(self,key,value):
        if key in self.nodes:
            node=self.nodes[key]; node.value=value; self._touch(node); return
        if not self.capacity: return
        if len(self.nodes)==self.capacity:
            victim=self.order.pop_left(); del self.nodes[victim.key]
        node=_CacheNode(key,value); self.nodes[key]=node; self.order.append(node)

from collections import OrderedDict

def normalize_solution_output(x):
    """For unordered collections of unordered integer combinations only.
    Multiplicity is preserved so duplicate answers remain detectable.
    """
    return tuple(sorted(tuple(sorted(row)) for row in x))

def assert_linked_structure(head):
    nodes=[];seen=set()
    while head is not None:
        assert id(head) not in seen, 'Cycle found'
        seen.add(id(head));nodes.append(head);head=head.next
    return nodes

def differential_test_lru(operations, capacity=2):
    cache=LRUCache(capacity);reference=OrderedDict();results=[]
    for op in operations:
        if op[0]=='put':
            _,key,value=op
            cache.put(key,value)
            if capacity:
                if key in reference:del reference[key]
                elif len(reference)==capacity:reference.popitem(last=False)
                reference[key]=value
        elif op[0]=='get':
            _,key=op
            expected=reference.get(key,-1)
            if key in reference:reference.move_to_end(key)
            actual=cache.get(key)
            assert actual==expected
            results.append(actual)
        else:raise ValueError('Unknown cache operation')
        nodes=[];node=cache.order.head.next;previous=cache.order.head
        while node is not cache.order.tail:
            assert node.prev is previous and node not in nodes
            nodes.append(node);previous,node=node,node.next
        assert cache.order.tail.prev is previous
        assert [(node.key,node.value) for node in nodes]==list(reference.items())
        assert set(nodes)==set(cache.nodes.values()) and cache.order.size==len(nodes)
    return results

def count_binary_search_iterations():
    rows=[]
    for n in [0,1,2,4,8,16,64,256,1024]:
        maximum=0
        for target in range(-1,n+1):
            left,right=0,n;count=0
            while left<right:
                middle=(left+right)//2;count+=1
                if middle<target:left=middle+1
                else:right=middle
            assert left==min(n,max(0,target))
            maximum=max(maximum,count)
        rows.append((n,maximum))
    return rows

## 正确性、前提与复杂度

独立参照实现维护同一接口但不同内部机制：OrderedDict 对比手写双链表；结构断言检查映射与链表一一对应。恢复测试比较对象引用而非仅比较值。二分的循环不变量是答案始终位于当前半开候选区间的边界范围内。

**代价：** 普通 LRU 操作仍期望 O(1)；本章诊断每一步扫描缓存，测试开销 O(操作数×容量)，不把诊断器的开销误称为算法开销。二分正式查询 O(log n)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['array length','maximum lower-bound loop iterations'],count_binary_search_iterations())
ops=[('put',1,10),('put',2,20),('get',1),('put',3,30),('get',2),('get',3)]
show_table(['operation trace','get results'],[(ops,differential_test_lru(ops))])

array length,maximum lower-bound loop iterations
0,0
1,1
2,2
4,3
8,4
16,5
64,7
256,9
1024,11


operation trace,get results
"[('put', 1, 10), ('put', 2, 20), ('get', 1), ('put', 3, 30), ('get', 2), ('get', 3)]","[10, -1, 30]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
import math,random
assert normalize_solution_output([[1,2],[3,4]])==normalize_solution_output([[4,3],[2,1]])
assert normalize_solution_output([[1,2],[1,2]])!=normalize_solution_output([[1,2]])
head=list_to_nodes([1,2,3]);before=assert_linked_structure(head)
links=[node.next for node in before]
def reverse(head):
    previous=None
    while head:
        nxt=head.next;head.next=previous;previous=head;head=nxt
    return previous
restored=reverse(reverse(head))
assert restored is head and assert_linked_structure(restored)==before
assert all(node.next is nxt for node,nxt in zip(before,links))
before[-1].next=head
try:assert_linked_structure(head)
except AssertionError:pass
else:raise AssertionError('Cycle went undetected')
before[-1].next=None
assert all(node.next is nxt for node,nxt in zip(before,links))
rng=random.Random(171)
operations=[('put',rng.randrange(7),rng.randrange(50)) if rng.randrange(2) else ('get',rng.randrange(7)) for _ in range(500)]
for capacity in [0,1,3]:
    assert differential_test_lru(operations,capacity)==differential_test_lru(operations,capacity)
for n,iterations in count_binary_search_iterations():
    assert iterations<=math.ceil(math.log2(n+1)) if n else iterations==0
print('N171: 所有本章断言通过')

N171: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 修复刻意注入的链表断链、LRU淘汰和二分边界错误。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 区分速度噪声与阶数变化。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 146. LRU Cache（comparison）
- 234. Palindrome Linked List（comparison）
- 34. Find First and Last Position of Element in Sorted Array（comparison）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。